# Celltypist and cellhint

In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd

## rna

In [2]:
adata_rna_ann = sc.read_h5ad('../5_matrix_merge/processed/ann_cell/ann_cell_rna.h5ad')
adata_rna_ann

AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [3]:
adata_rna_no_ann = sc.read_h5ad('../5_matrix_merge/processed/no_annotation/filter_cell_rna.h5ad')
adata_rna_no_ann

AnnData object with n_obs × n_vars = 349124 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [6]:
adata_rna_no_ann.obs = adata_rna_no_ann.obs[['dataset','barcode', 'sample']]

In [7]:
del adata_rna_no_ann.var

In [9]:
adata_rna_no_ann.obs['cellType'] = 'notAvailable'

/tmp/ipykernel_3122665/4067322133.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adata_rna_no_ann.obs['cellType'] = 'notAvailable'


In [12]:
adata_rna_no_ann

AnnData object with n_obs × n_vars = 349124 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [16]:
adata_rna = ad.concat([adata_rna_ann, adata_rna_no_ann], join='outer', merge='unique', axis=0)

In [17]:
adata_rna

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [19]:
adata_rna.write_h5ad('./processed/adata_rna.h5ad', compression='gzip')

## 10x to ann

In [3]:
adata = sc.read_h5ad('./processed/adata_rna.h5ad')
adata

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [24]:
adata.X

<2462628x36601 sparse matrix of type '<class 'numpy.float32'>'
	with 3114721489 stored elements in Compressed Sparse Row format>

In [12]:
adata[:1000000,:].X

<1000000x36601 sparse matrix of type '<class 'numpy.float32'>'
	with 1671796325 stored elements in Compressed Sparse Row format>

In [13]:
1671796325/2147483647

0.7784908291783607

In [20]:
def anndata_to_10x(adata:sc.AnnData, _dir):
    from scipy.io import mmwrite
    from os.path import join, exists
    from os import makedirs
    if not exists(_dir):
        makedirs(_dir)
    X = adata[:1000000,:].X.T.tocoo()
    mmwrite(join(_dir, 'matrix.mtx'), X)
    var = pd.DataFrame(adata[:1000000,:].var.index, columns=['gene_symbol'])
    var.to_csv(join(_dir, 'genes.tsv'), sep='\t', header=True, index=False)
    adata[:1000000,:].obs.to_csv(join(_dir, 'barcodes.tsv'), sep='\t', header=True, index=True)

anndata_to_10x(adata, './processed/scanpy_10x/part_1/')

In [21]:
adata[1000000:,:].X

<1462628x36601 sparse matrix of type '<class 'numpy.float32'>'
	with 1442925164 stored elements in Compressed Sparse Row format>

In [22]:
1442925164/2147483647

0.6719143896698553

In [23]:
1442925164+1671796325

3114721489

In [25]:
def anndata_to_10x(adata:sc.AnnData, _dir):
    from scipy.io import mmwrite
    from os.path import join, exists
    from os import makedirs
    if not exists(_dir):
        makedirs(_dir)
    X = adata[1000000:,:].X.T.tocoo()
    mmwrite(join(_dir, 'matrix.mtx'), X)
    var = pd.DataFrame(adata[1000000:,:].var.index, columns=['gene_symbol'])
    var.to_csv(join(_dir, 'genes.tsv'), sep='\t', header=True, index=False)
    adata[1000000:,:].obs.to_csv(join(_dir, 'barcodes.tsv'), sep='\t', header=True, index=True)

anndata_to_10x(adata, './processed/scanpy_10x/part_2/')

## atac

In [2]:
adata_atac = sc.read_h5ad('../5_matrix_merge/processed/all_cell/atac.h5ad')
adata_atac

AnnData object with n_obs × n_vars = 2556326 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [4]:
adata_rna = sc.read_h5ad('./processed/adata_rna.h5ad')

In [9]:
adata_atac = adata_atac[adata_rna.obs_names.tolist()]
adata_atac

View of AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [10]:
adata_atac.obs['cellType'] = adata_rna.obs['cellType'].tolist()

/tmp/ipykernel_460093/3635731153.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_atac.obs['cellType'] = adata_rna.obs['cellType'].tolist()


In [11]:
adata_atac

AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [12]:
adata_atac.write_h5ad('./processed/adata_atac.h5ad', compression='gzip')

## celltypist_cellhint

In [ ]:
# import celltypist
# import cellhint
# import os
# import scanpy as sc
# os.chdir('/pool1/qiaolibo/R_Project/peak_gene/z_others/6_ann_batch')

# adata = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(adata, target_sum = 1e4)
# sc.pp.log1p(adata)

# adata = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True).to_adata()

# adata.write_h5ad('./processed/adata_rna_celltypist.h5ad', compression='gzip')

# cellhint.integrate(adata, 'sample', 'majority_voting')

# sc.tl.umap(adata)

# adata.write_h5ad('./processed/adata_rna_celltypist_cellhint.h5ad', compression='gzip')

In [3]:
test = sc.read_h5ad('./processed/adata_rna_celltypist.h5ad')

In [4]:
test

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    uns: 'log1p', 'neighbors', 'over_clustering'
    obsm: 'X_pca'
    obsp: 'connectivities', 'distances'

In [3]:
adata = sc.read_h5ad('./processed/adata_rna_celltypist_cellhint.h5ad')
adata

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    uns: 'log1p', 'neighbors', 'over_clustering', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [5]:
adata.obs.to_csv('processed/cellTypist.csv')

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

## 添加level

In [4]:
import os
os.chdir('/pool1/qiaolibo/R_Project/peak_gene/z_others/')

## cellhint majority_voting

In [5]:
data = sc.read_h5ad('6_ann_batch/processed/adata_rna_celltypist_cellhint.h5ad')
data

In [5]:
cell_type = pd.read_csv('./6_ann_batch/processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell_type

/tmp/ipykernel_1894369/3943989773.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)


,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
93,others,Cycling cells,Cycling DCs
94,others,Cycling cells,Cycling gamma-delta T cells
95,others,Cycling cells,Cycling monocytes
96,others,Cycling cells,Cycling NK cells


In [6]:
all = pd.merge(data.obs, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = data.obs_names.tolist()
all

,dataset,barcode,sample,cellType,predicted_labels,over_clustering,majority_voting,conf_score,level_1,level_2,level_3
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,AAACAGCCAACTGGCT,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.511893,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,AAACATGCACAATTAC,Chen_Nat_Methods_2022-lane1,Treg,Tem/Effector helper T cells,264,Tcm/Naive helper T cells,0.011004,Lymphoid,T cells,Tem/Effector helper T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,AAACATGCAGCACGTT,Chen_Nat_Methods_2022-lane1,Th17,CD16- NK cells,48,Tem/Effector helper T cells,0.022152,Lymphoid,ILC,CD16- NK cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,AAACATGCATATTGAC,Chen_Nat_Methods_2022-lane1,Th1,Tem/Trm cytotoxic T cells,219,Tcm/Naive helper T cells,0.953910,Lymphoid,T cells,Tem/Trm cytotoxic T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,AAACCAACAATAGCAA,Chen_Nat_Methods_2022-lane1,TCM,Tcm/Naive helper T cells,70,Tcm/Naive helper T cells,0.222813,Lymphoid,T cells,Tcm/Naive helper T cells
...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7,TTTGTGTTCGTAATCA,Weng_Nature_2024_7-1,notAvailable,Small pre-B cells,238,Small pre-B cells,0.020311,Lymphoid,B-cell lineage,Small pre-B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7,TTTGTTGGTAGGATTT,Weng_Nature_2024_7-1,notAvailable,Small pre-B cells,238,Small pre-B cells,0.621049,Lymphoid,B-cell lineage,Small pre-B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7,TTTGTTGGTCATTACC,Weng_Nature_2024_7-1,notAvailable,Memory B cells,95,Naive B cells,0.999996,Lymphoid,B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7,TTTGTTGGTGTTAAAC,Weng_Nature_2024_7-1,notAvailable,Tem/Temra cytotoxic T cells,43,Tem/Trm cytotoxic T cells,0.229349,Lymphoid,T cells,Tem/Temra cytotoxic T cells


In [7]:
data.obs = all

In [ ]:
sc.pl.umap(data, color='level_2')

In [12]:
data.write_h5ad('6_ann_batch/processed/adata_rna_celltypist_cellhint.h5ad', compression='gzip')

## cellhint predicted_labels

In [5]:
data = sc.read_h5ad('6_ann_visual/processed/adata_rna_cellhint_1.h5ad')
data

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    uns: 'log1p', 'neighbors', 'over_clustering', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [6]:
cell_type = pd.read_csv('./6_ann_visual/processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell_type

/tmp/ipykernel_2710090/2124838903.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)


,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
93,others,Cycling cells,Cycling DCs
94,others,Cycling cells,Cycling gamma-delta T cells
95,others,Cycling cells,Cycling monocytes
96,others,Cycling cells,Cycling NK cells


In [7]:
all = pd.merge(data.obs, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = data.obs_names.tolist()
all

,dataset,barcode,sample,cellType,predicted_labels,over_clustering,majority_voting,conf_score,level_1,level_2,level_3
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,AAACAGCCAACTGGCT,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.511893,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,AAACATGCACAATTAC,Chen_Nat_Methods_2022-lane1,Treg,Tem/Effector helper T cells,264,Tcm/Naive helper T cells,0.011004,Lymphoid,T cells,Tem/Effector helper T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,AAACATGCAGCACGTT,Chen_Nat_Methods_2022-lane1,Th17,CD16- NK cells,48,Tem/Effector helper T cells,0.022152,Lymphoid,ILC,CD16- NK cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,AAACATGCATATTGAC,Chen_Nat_Methods_2022-lane1,Th1,Tem/Trm cytotoxic T cells,219,Tcm/Naive helper T cells,0.953910,Lymphoid,T cells,Tem/Trm cytotoxic T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,AAACCAACAATAGCAA,Chen_Nat_Methods_2022-lane1,TCM,Tcm/Naive helper T cells,70,Tcm/Naive helper T cells,0.222813,Lymphoid,T cells,Tcm/Naive helper T cells
...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7,TTTGTGTTCGTAATCA,Weng_Nature_2024_7-1,notAvailable,Small pre-B cells,238,Small pre-B cells,0.020311,Lymphoid,B-cell lineage,Small pre-B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7,TTTGTTGGTAGGATTT,Weng_Nature_2024_7-1,notAvailable,Small pre-B cells,238,Small pre-B cells,0.621049,Lymphoid,B-cell lineage,Small pre-B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7,TTTGTTGGTCATTACC,Weng_Nature_2024_7-1,notAvailable,Memory B cells,95,Naive B cells,0.999996,Lymphoid,B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7,TTTGTTGGTGTTAAAC,Weng_Nature_2024_7-1,notAvailable,Tem/Temra cytotoxic T cells,43,Tem/Trm cytotoxic T cells,0.229349,Lymphoid,T cells,Tem/Temra cytotoxic T cells


In [8]:
data.obs = all

In [ ]:
sc.pl.umap(data, color='level_1')

In [ ]:
sc.pl.umap(data, color='level_2')

In [13]:
data.obs['level_3'].value_counts()

level_3
Tcm/Naive helper T cells       759471
Late erythroid                 295023
Plasma cells                   235012
Classical monocytes            164203
Tem/Temra cytotoxic T cells     79759
                                ...  
Cycling B cells                    42
ILC                                40
pDC precursor                      35
ILC1                               18
Cycling gamma-delta T cells         7
Name: count, Length: 98, dtype: int64

In [14]:
data.obs['level_2'].value_counts()

level_2
T cells                       1163570
Erythroid                      309961
Plasma cells                   237920
Monocytes                      190481
ILC                            138938
B cells                        123378
Epithelial cells                67408
HSC/MPP                         54837
Macrophages                     46371
Double-positive thymocytes      19886
B-cell lineage                  17557
Fibroblasts                     17233
Megakaryocyte precursor         14779
Double-negative thymocytes      11944
pDC                             10314
DC                              10160
Endothelial cells                8518
ETP                              6426
Mast cells                       2210
MNP                              2147
Granulocytes                     1833
Early MK                         1750
Cycling cells                    1456
Mono-mac                         1128
Monocyte precursor                678
DC precursor                      488
Eryt

In [16]:
data.obs['level_1'].value_counts()

level_1
Lymphoid    1719619
Myeloid      593557
others        94615
HSC/MPP       54837
Name: count, dtype: int64

# snapatac2

In [1]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import pandas as pd
import muon

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## rna

### spectral 1

In [7]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = rna.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = rna.obs_names.tolist()
# all

# rna.obs = all

# rna.write_h5ad('./processed/adata_rna_snap_umap.h5ad')

In [28]:
rna = sc.read_h5ad('./processed/adata_rna_snap_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 3000
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [29]:
rna.obsm['X_spectral'].dtype

dtype('float64')

In [ ]:
sc.pl.umap(rna, color="level_1")

### spectral 2

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]

# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = rna.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = rna.obs_names.tolist()
# all

# rna.obs = all

# rna.write_h5ad('./processed/adata_rna_snap2_umap.h5ad')

In [6]:
rna = sc.read_h5ad('./processed/adata_rna_snap2_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [4]:
import harmonypy as hm

In [5]:
meta_data = rna.obs
vars_use = ['sample']
data_mat = rna.obsm['X_spectral']
ho = hm.run_harmony(data_mat, meta_data, vars_use, max_iter_harmony=20)

2024-11-22 21:51:02,201 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2024-11-22 21:51:02 - INFO - Computing initial centroids with sklearn.KMeans...
2024-11-22 21:52:35,694 - harmonypy - INFO - sklearn.KMeans initialization complete.
2024-11-22 21:52:35 - INFO - sklearn.KMeans initialization complete.
2024-11-22 21:52:41,822 - harmonypy - INFO - Iteration 1 of 20
2024-11-22 21:52:41 - INFO - Iteration 1 of 20


KeyboardInterrupt: 

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='technology')

## norm

In [ ]:
# adata = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# adata.raw = adata
# adata = adata[:, adata.var.highly_variable]
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, svd_solver='arpack')

# sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
# sc.tl.umap(adata)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = adata.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = adata.obs_names.tolist()
# all

# adata.obs = all

# adata.write_h5ad('./processed/adata_rna_umap.h5ad')

In [2]:
rna = sc.read_h5ad('./processed/adata_rna_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [4]:
pca = pd.DataFrame(rna.obsm['X_pca'])
pca

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,-5.925003,-3.944832,-0.824258,-1.331650,1.381276,-2.049721,-3.999266,6.992177,-0.099963,-0.039602,...,1.746100,-2.880136,0.650043,0.536532,-0.264131,0.374774,0.829798,0.250654,-0.183003,-1.412524
1,-4.947170,-4.379195,-0.526318,-1.062181,0.083710,0.137350,-2.657282,4.311140,0.764121,1.152642,...,-1.179102,-1.649010,0.952981,0.976905,1.951936,0.495050,-0.025391,-0.467815,-0.375358,-0.299769
2,-4.994188,-5.452335,-1.837202,-0.709865,1.150625,-1.757070,-4.811746,6.400196,-0.293773,0.095057,...,-1.944518,-1.662315,1.802830,2.051769,2.067444,-0.065966,-1.877600,-2.023523,-0.449247,1.068079
3,-5.643923,-4.585869,-1.793504,-1.934368,1.672878,-0.804959,-3.238342,7.422114,-0.997438,-0.086153,...,-1.756791,-1.727501,2.238896,2.091069,0.314278,0.726498,-1.688851,-0.450971,-0.086558,-0.186492
4,-4.361089,-4.195010,-0.654042,-1.243502,1.462886,-1.007589,-3.518795,3.878448,0.153096,-1.372369,...,-3.018039,-2.176234,0.373872,4.048567,-1.046607,0.492838,-2.029928,-1.320077,-0.544095,-0.479906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,-6.768851,7.380241,7.615761,20.068848,0.907372,-4.118711,1.912593,2.203979,1.319968,-6.988191,...,1.567266,-1.575111,0.429493,3.200652,-0.952867,0.719388,-1.450721,-1.394291,2.124093,3.942337
2462624,-11.940614,7.912078,7.325287,10.047938,9.335398,-2.664724,2.401935,3.940327,4.492854,-6.146708,...,1.149761,-1.493250,-1.415791,1.329392,0.674990,0.969388,-1.643811,-0.203857,0.370678,1.058613
2462625,-7.295238,5.055811,2.526584,15.369665,-0.560737,-4.517577,1.160833,3.650206,-2.461400,3.039872,...,-0.752585,-1.032686,-0.487358,0.127705,0.879010,-0.237720,-1.459671,-1.896807,-1.762699,3.509446
2462626,-7.746551,-2.678474,-0.175563,-3.540229,3.378915,-2.169612,8.003254,1.226881,-2.107147,1.156979,...,-0.334460,0.594009,0.315604,-0.199386,1.738631,0.514833,-0.780161,0.626585,-0.357675,-1.497050


In [5]:
pca.to_csv('./processed/rna_pca.csv')

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="technology")

## atac 

### spectral

In [10]:
# rna = sc.read_h5ad('./processed/adata_atac.h5ad')
# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = rna.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = rna.obs_names.tolist()
# all

# rna.obs = all

# rna.write_h5ad('./processed/adata_atac_umap.h5ad')

In [12]:
atac = sc.read_h5ad('./processed/adata_atac_umap.h5ad')
atac

AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    uns: 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [ ]:
sc.pl.umap(atac, color="level_2")

In [ ]:
sc.pl.umap(atac, color="technology")

### ti-idf

In [2]:
atac = sc.read_h5ad('./processed/adata_atac.h5ad')
atac

AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [ ]:
muon.atac.tl.lsi(atac)
atac

In [2]:
embedding = np.load('./processed/atac_lsi.npy')
embedding

array([[-0.41459064,  0.44059818,  0.06066888, ..., -0.19684204,
        -1.02128552, -1.66321146],
       [-0.05459714,  0.55399356, -0.08399562, ..., -1.48162652,
         0.81062412, -2.74223016],
       [-0.5609467 ,  0.38615726, -0.12688975, ..., -0.59777518,
         0.85085502, -1.46410062],
       ...,
       [ 0.88832803, -0.61041967, -1.08786507, ..., -0.52908774,
        -0.10638634,  1.97008038],
       [ 0.58319716,  0.60374567,  0.3670135 , ...,  0.43286202,
         3.08916304,  1.11608924],
       [ 1.07238242, -1.89840479, -2.43373014, ..., -0.48802436,
         0.5847141 ,  2.18865582]])

In [ ]:
b = pd.read_

In [3]:
a = pd.DataFrame(embedding)
a

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,-0.414591,0.440598,0.060669,0.077042,-0.614596,0.563807,0.257098,-0.432166,-3.459507,-1.182690,...,-1.954424,-1.539233,1.234186,-0.329961,-3.329448,-0.622604,1.378115,-0.196842,-1.021286,-1.663211
1,-0.054597,0.553994,-0.083996,-0.209268,-1.511775,0.867192,0.078537,-0.401550,-5.172728,-1.835391,...,-1.351694,-3.117907,1.990113,0.385460,-3.941945,-1.631970,2.921233,-1.481627,0.810624,-2.742230
2,-0.560947,0.386157,-0.126890,-0.263929,-1.381435,0.471572,-0.216481,-1.315798,-3.217611,-0.144057,...,-0.700923,0.135793,0.556834,-0.262756,-0.325134,0.979129,0.511120,-0.597775,0.850855,-1.464101
3,-0.285761,0.438576,0.088747,-0.324684,-1.584955,0.228338,-0.332140,-1.613588,-5.201733,-0.707897,...,-2.258918,1.760940,-0.070585,0.777408,-1.196876,2.191314,-1.725290,-1.262641,-3.579710,-0.667843
4,-0.303495,0.601150,-0.090293,-0.128129,-0.948027,1.486466,0.136521,-0.421280,-4.225236,-1.507518,...,1.377766,-1.124739,0.398383,1.785150,-1.167563,-1.714722,-0.140318,-0.218995,-1.138308,-2.114346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,0.385265,-1.354666,-1.335801,2.656147,0.474883,-0.052079,0.308512,0.493667,0.088864,1.530825,...,2.515132,1.846518,1.918099,-0.839754,-0.523032,-2.544367,0.938338,2.249888,0.977743,0.448289
2462624,1.218185,-0.456078,-1.365595,2.629032,-0.466292,-1.432098,-0.044475,0.506532,0.835383,3.266482,...,1.952893,0.143896,0.892021,-2.285338,1.430399,0.001490,1.129607,1.601702,0.635790,0.399995
2462625,0.888328,-0.610420,-1.087865,4.427346,-0.042616,-0.570754,0.754890,-1.301833,1.045001,-1.494597,...,-1.143286,-0.902853,-0.844487,0.097747,0.091739,0.233523,1.539554,-0.529088,-0.106386,1.970080
2462626,0.583197,0.603746,0.367013,-0.130307,-1.623049,-1.829437,-0.164565,1.048016,-0.034788,0.257474,...,0.394482,1.484157,-0.301413,-0.135226,0.393673,0.048490,0.012560,0.432862,3.089163,1.116089


In [4]:
a.to_csv('./processed/atac_lsi.csv')

## joint

### spectral

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# snap.tl.spectral(rna, features=None)

In [ ]:
# atac = sc.read_h5ad('./processed/adata_atac.h5ad')
# snap.tl.spectral(atac, features=None)
# snap.tl.umap(atac)

In [12]:
# assert (rna.obs_names == atac.obs_names).all()
# embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]
# np.save('embedding.npy', embedding)

# rna = sc.read_h5ad('./processed/adata_rna.h5ad')
# embedding = np.load('./processed/embedding.npy')
# rna.obsm['rna_atac'] = embedding
# snap.tl.umap(rna, use_rep='rna_atac')
# np.save('./processed/umap.npy', rna.obsm['X_umap'])

In [4]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')

In [12]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()

cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
all

,dataset,barcode,sample,cellType,predicted_labels,over_clustering,majority_voting,conf_score,tissue,disease_name,disease,age,age_stage,sex,technology,others,sample_
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,AAACAGCCAACTGGCT,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.511893,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,AAACATGCACAATTAC,Chen_Nat_Methods_2022-lane1,Treg,Tem/Effector helper T cells,264,Tcm/Naive helper T cells,0.011004,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,AAACATGCAGCACGTT,Chen_Nat_Methods_2022-lane1,Th17,CD16- NK cells,48,Tem/Effector helper T cells,0.022152,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,AAACATGCATATTGAC,Chen_Nat_Methods_2022-lane1,Th1,Tem/Trm cytotoxic T cells,219,Tcm/Naive helper T cells,0.953910,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,AAACCAACAATAGCAA,Chen_Nat_Methods_2022-lane1,TCM,Tcm/Naive helper T cells,70,Tcm/Naive helper T cells,0.222813,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7,TTTGTGTTCGTAATCA,Weng_Nature_2024_7-1,notAvailable,Small pre-B cells,238,Small pre-B cells,0.020311,bone marrow,Healthy,Healthy,81/85/69,notAvailable,notAvailable,ReDeeM,NaN,Weng_Nature_2024_7-1
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7,TTTGTTGGTAGGATTT,Weng_Nature_2024_7-1,notAvailable,Small pre-B cells,238,Small pre-B cells,0.621049,bone marrow,Healthy,Healthy,81/85/69,notAvailable,notAvailable,ReDeeM,NaN,Weng_Nature_2024_7-1
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7,TTTGTTGGTCATTACC,Weng_Nature_2024_7-1,notAvailable,Memory B cells,95,Naive B cells,0.999996,bone marrow,Healthy,Healthy,81/85/69,notAvailable,notAvailable,ReDeeM,NaN,Weng_Nature_2024_7-1
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7,TTTGTTGGTGTTAAAC,Weng_Nature_2024_7-1,notAvailable,Tem/Temra cytotoxic T cells,43,Tem/Trm cytotoxic T cells,0.229349,bone marrow,Healthy,Healthy,81/85/69,notAvailable,notAvailable,ReDeeM,NaN,Weng_Nature_2024_7-1


In [22]:
rna.obs = all

In [8]:
umap = np.load('./processed/umap.npy')

In [9]:
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="dataset")

In [ ]:
sc.pl.umap(rna, color="technology")

### spectral 2

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]

# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

In [3]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')

In [5]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()

cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
all

/tmp/ipykernel_3399795/580747303.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)


,dataset,barcode,sample,cellType,predicted_labels,over_clustering,majority_voting,conf_score,tissue,disease_name,disease,age,age_stage,sex,technology,others,sample_,level_1,level_2,level_3
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,AAACAGCCAACTGGCT,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.511893,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,AAACCAACAGGCTTCG,Chen_Nat_Methods_2022-lane1,Act,Regulatory T cells,70,Tcm/Naive helper T cells,0.040097,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,AAACCGGCACCATATG,Chen_Nat_Methods_2022-lane1,Th2,Regulatory T cells,70,Tcm/Naive helper T cells,0.044226,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,AACCGGCTCAATTACG,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.999846,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,AACTTAGTCACTTTAC,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,12,Tem/Effector helper T cells,0.957420,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Hu_BioRxiv_2023,T128.R1.185.R2.034.R3.012.P1.07,Hu_BioRxiv_2023-donor6,early-Ery,Cycling gamma-delta T cells,51,Late erythroid,1.000000,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor6,others,Cycling cells,Cycling gamma-delta T cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Hu_BioRxiv_2023,T124.R1.032.R2.058.R3.024.P1.04,Hu_BioRxiv_2023-donor7,CD8,Cycling gamma-delta T cells,166,Plasma cells,0.676814,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor7,others,Cycling cells,Cycling gamma-delta T cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Hu_BioRxiv_2023,T124.R1.068.R2.111.R3.044.P1.04,Hu_BioRxiv_2023-donor7,early-Ery,Cycling gamma-delta T cells,181,Epithelial cells,0.999090,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor7,others,Cycling cells,Cycling gamma-delta T cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Hu_BioRxiv_2023,T124.R1.177.R2.063.R3.037.P1.04,Hu_BioRxiv_2023-donor7,early-Ery,Cycling gamma-delta T cells,181,Epithelial cells,1.000000,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor7,others,Cycling cells,Cycling gamma-delta T cells


In [6]:
rna.obs = all

In [7]:
umap = np.load('./processed/umap_snap2.npy')
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="technology")

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="dataset")

### norm 

In [33]:
atac = sc.read_h5ad('./processed/adata_atac_umap.h5ad')

rna = sc.read_h5ad('./processed/adata_rna_umap.h5ad')

In [43]:
rna.obsm['X_spectral'] = rna.obsm['X_pca'][:, :30].astype(np.float64)

In [46]:
rna.obsm['X_spectral']

array([[-5.92500257, -3.94483185, -0.82425785, ...,  0.38249373,
        -0.645078  ,  0.1471799 ],
       [-4.94716978, -4.37919474, -0.52631766, ..., -0.41722521,
         1.28118968,  0.82659155],
       [-4.99418831, -5.45233536, -1.83720219, ...,  0.24899279,
         1.68344164,  0.64879847],
       ...,
       [-7.29523849,  5.05581141,  2.52658367, ..., -2.47483158,
         0.44544607, -1.19943845],
       [-7.74655104, -2.67847395, -0.17556299, ...,  1.78517628,
        -1.1852206 ,  0.87337762],
       [-9.47505951, 15.38859272, 16.68761826, ...,  0.05537876,
        -0.57865888, -3.11202598]])

In [47]:
atac.obsm['X_spectral']

array([[ 5.33250323e-04,  7.08676888e-05, -2.87095204e-05, ...,
         4.91645160e-05, -3.58209658e-05, -3.01902537e-05],
       [ 5.79639631e-04,  7.78894517e-05, -5.82660648e-05, ...,
        -3.80681532e-06, -1.97840887e-05, -3.69559087e-05],
       [ 5.07815382e-04,  5.65700434e-05, -7.02739720e-05, ...,
         2.21630948e-05,  4.61693656e-06, -2.91862578e-05],
       ...,
       [ 7.25739307e-04, -1.46204816e-04, -1.71381810e-04, ...,
        -4.10395312e-05,  2.16440305e-05,  8.05051841e-05],
       [ 7.09763088e-04,  8.84136911e-05, -1.90918500e-06, ...,
         7.72210135e-07, -1.77290338e-05, -1.25886295e-05],
       [ 7.62368553e-04, -3.75562923e-04, -2.74799328e-04, ...,
        -6.08777579e-05,  3.20974722e-05, -4.52158855e-05]])

In [44]:
embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]

2024-11-20 01:46:13 - INFO - Compute normalized views...
thread '<unnamed>' panicked at snapatac2-python/src/embedding.rs:411:13:
called `Result::unwrap()` on an `Err` value: Cannot convert Array(F32) to f64 CsrMatrix


PanicException: called `Result::unwrap()` on an `Err` value: Cannot convert Array(F32) to f64 CsrMatrix

## harmony

In [5]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [2]:
import scanpy.external as sce
sce.pp.harmony_integrate(rna, 'sample')

In [6]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()

cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
all

/tmp/ipykernel_1722677/580747303.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)


,dataset,barcode,sample,cellType,predicted_labels,over_clustering,majority_voting,conf_score,tissue,disease_name,disease,age,age_stage,sex,technology,others,sample_,level_1,level_2,level_3
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,AAACAGCCAACTGGCT,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.511893,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,AAACCAACAGGCTTCG,Chen_Nat_Methods_2022-lane1,Act,Regulatory T cells,70,Tcm/Naive helper T cells,0.040097,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,AAACCGGCACCATATG,Chen_Nat_Methods_2022-lane1,Th2,Regulatory T cells,70,Tcm/Naive helper T cells,0.044226,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,AACCGGCTCAATTACG,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,67,Regulatory T cells,0.999846,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,AACTTAGTCACTTTAC,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,12,Tem/Effector helper T cells,0.957420,blood,Healthy,healthy,notAvailable,notAvailable,notAvailable,NEAT-seq,NaN,Chen_Nat_Methods_2022-lane1,Lymphoid,T cells,Regulatory T cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Hu_BioRxiv_2023,T128.R1.185.R2.034.R3.012.P1.07,Hu_BioRxiv_2023-donor6,early-Ery,Cycling gamma-delta T cells,51,Late erythroid,1.000000,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor6,others,Cycling cells,Cycling gamma-delta T cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Hu_BioRxiv_2023,T124.R1.032.R2.058.R3.024.P1.04,Hu_BioRxiv_2023-donor7,CD8,Cycling gamma-delta T cells,166,Plasma cells,0.676814,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor7,others,Cycling cells,Cycling gamma-delta T cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Hu_BioRxiv_2023,T124.R1.068.R2.111.R3.044.P1.04,Hu_BioRxiv_2023-donor7,early-Ery,Cycling gamma-delta T cells,181,Epithelial cells,0.999090,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor7,others,Cycling cells,Cycling gamma-delta T cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Hu_BioRxiv_2023,T124.R1.177.R2.063.R3.037.P1.04,Hu_BioRxiv_2023-donor7,early-Ery,Cycling gamma-delta T cells,181,Epithelial cells,1.000000,bone marrow,Healthy,Healthy,notAvailable,notAvailable,notAvailable,SHARE-seq,NaN,Hu_BioRxiv_2023-donor7,others,Cycling cells,Cycling gamma-delta T cells


In [7]:
rna.obs = all

In [8]:
umap = np.load('./processed/umap_harmony.npy')
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="level_1")

In [ ]:
sc.pl.umap(rna, color="technology")

## cellhint

In [13]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [14]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()
cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
rna.obs = all

/tmp/ipykernel_1722677/2310988750.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)


In [16]:
umap = np.load('./processed/umap_cellhint.npy')
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="technology")

# Multiple algorithms and visual

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

## spectral

### rna_raw

In [36]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [37]:
embedding = pd.DataFrame(rna.obsm['X_spectral'])
embedding

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,-2.868716e-08,-6.385926e-09,-2.261692e-09,-6.990828e-09,0.000690,9.367233e-09,-4.987228e-09,7.518664e-09,-0.000003,8.952668e-09,...,0.000002,-0.000096,-0.000401,0.000018,-0.000033,0.000058,-0.000026,-2.002737e-05,-0.000160,-0.000009
1,-2.788127e-08,-6.206531e-09,-2.198156e-09,-6.794439e-09,0.000671,9.104086e-09,-4.847125e-09,7.307447e-09,-0.000003,8.701167e-09,...,-0.000067,0.000012,-0.000286,-0.000017,0.000006,0.000122,0.000042,2.523264e-05,-0.000184,-0.000044
2,-2.750625e-08,-6.123049e-09,-2.168589e-09,-6.703050e-09,0.000662,8.981631e-09,-4.781929e-09,7.209158e-09,-0.000003,8.584131e-09,...,0.000015,-0.000091,-0.000360,-0.000014,0.000057,-0.000085,-0.000013,-1.698742e-05,0.000116,-0.000025
3,-2.723881e-08,-6.063516e-09,-2.147504e-09,-6.637877e-09,0.000656,8.894303e-09,-4.735435e-09,7.139064e-09,-0.000003,8.500669e-09,...,0.000005,-0.000049,-0.000359,0.000018,-0.000119,-0.000053,-0.000114,-2.372800e-07,0.000090,0.000002
4,-2.755017e-08,-6.132827e-09,-2.172052e-09,-6.713754e-09,0.000663,8.995973e-09,-4.789565e-09,7.220670e-09,-0.000003,8.597839e-09,...,-0.000034,-0.000032,-0.000188,0.000095,-0.000080,0.000011,-0.000045,1.844008e-05,-0.000037,-0.000060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,-2.794657e-08,-6.221067e-09,-2.203304e-09,-6.810353e-09,0.000673,9.125409e-09,-4.858478e-09,7.324563e-09,-0.000003,8.721547e-09,...,0.000384,-0.000183,-0.000046,0.000066,-0.000124,-0.000103,0.000176,-3.225822e-04,-0.000020,0.000050
2462624,-3.418167e-08,-7.609036e-09,-2.694878e-09,-8.329796e-09,0.000823,1.116136e-08,-5.942442e-09,8.958729e-09,-0.000003,1.066739e-08,...,0.000242,-0.000102,-0.000106,0.000068,-0.000167,0.000015,0.000230,-3.123901e-04,0.000041,0.000073
2462625,-2.877220e-08,-6.404857e-09,-2.268397e-09,-7.011552e-09,0.000693,9.395002e-09,-5.002013e-09,7.540953e-09,-0.000003,8.979208e-09,...,0.000458,-0.000311,-0.000121,-0.000150,-0.000045,-0.000189,-0.000039,-1.063617e-05,0.000048,0.000025
2462626,-2.949273e-08,-6.565250e-09,-2.325203e-09,-7.187139e-09,0.000710,9.630276e-09,-5.127275e-09,7.729797e-09,-0.000003,9.204070e-09,...,0.000050,-0.000238,0.000056,-0.000159,-0.000176,-0.000213,0.000031,-1.054805e-04,0.000024,0.000051


In [38]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/rna/embedding_raw.csv')

In [39]:
umap = pd.DataFrame(rna.obsm['X_umap'])
umap

,0,1
0,6.353551,-0.489735
1,6.759045,-0.637842
2,5.711500,-0.324390
3,5.205359,-0.349282
4,6.080683,-1.209088
...,...,...
2462623,6.128721,8.793364
2462624,6.985214,8.370503
2462625,5.478915,9.667448
2462626,13.174591,4.848317


In [40]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/rna/umap_raw.csv')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### rna_harmony

In [87]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [91]:
umap = pd.read_csv('processed/z_embedding_umap/multi_spectral/rna/umap_harmony.csv', index_col=0)
umap

,0,1
0,10.183611,6.789410
1,12.145048,5.504837
2,11.093887,7.140558
3,10.707772,7.229662
4,14.327003,7.257714
...,...,...
2462623,4.238579,-6.533208
2462624,7.947500,-4.705008
2462625,6.438507,-5.706579
2462626,15.198705,-2.143763


In [92]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_raw

In [27]:
rna = sc.read_h5ad('processed/adata_atac_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    uns: 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [28]:
embedding = pd.DataFrame(rna.obsm['X_spectral'])
embedding

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.000533,0.000071,-0.000029,0.000084,-0.000055,-0.000056,0.000074,-0.000037,-0.000157,-0.000048,...,-0.000011,0.000002,0.000011,-0.000042,-0.000021,1.310251e-05,0.000011,4.916452e-05,-0.000036,-0.000030
1,0.000580,0.000078,-0.000058,0.000126,-0.000134,-0.000086,0.000067,-0.000062,-0.000195,-0.000049,...,-0.000033,0.000034,-0.000116,-0.000018,0.000022,4.073402e-05,-0.000067,-3.806815e-06,-0.000020,-0.000037
2,0.000508,0.000057,-0.000070,0.000128,-0.000156,-0.000063,0.000022,0.000045,-0.000187,-0.000015,...,-0.000022,0.000023,0.000042,-0.000014,-0.000030,9.420141e-07,0.000018,2.216309e-05,0.000005,-0.000029
3,0.000555,0.000066,-0.000034,0.000134,-0.000168,-0.000037,0.000017,0.000016,-0.000268,-0.000025,...,-0.000017,0.000038,-0.000005,0.000007,-0.000033,-2.252402e-05,0.000032,6.228624e-05,0.000003,-0.000043
4,0.000542,0.000101,-0.000053,0.000102,-0.000079,-0.000158,0.000089,-0.000047,-0.000163,-0.000058,...,0.000019,0.000034,-0.000040,-0.000026,0.000011,6.811120e-06,-0.000057,1.437318e-05,0.000072,0.000010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,0.000587,-0.000271,-0.000161,0.000087,0.000210,0.000041,-0.000002,-0.000006,0.000054,0.000126,...,-0.000024,0.000080,-0.000021,0.000012,0.000037,2.207128e-06,-0.000025,1.981535e-05,-0.000053,-0.000037
2462624,0.000804,-0.000111,-0.000184,0.000130,0.000138,0.000148,-0.000044,0.000015,0.000080,0.000198,...,-0.000089,0.000063,0.000033,-0.000017,-0.000022,-2.910015e-05,-0.000048,2.257797e-05,-0.000061,-0.000008
2462625,0.000726,-0.000146,-0.000171,0.000220,0.000280,0.000114,0.000039,0.000118,0.000052,-0.000121,...,0.000050,-0.000014,-0.000058,-0.000041,0.000014,-1.783599e-05,0.000016,-4.103953e-05,0.000022,0.000081
2462626,0.000710,0.000088,-0.000002,0.000099,-0.000145,0.000167,-0.000035,-0.000046,0.000040,0.000007,...,0.000012,0.000019,0.000046,-0.000008,-0.000009,2.105452e-05,0.000019,7.722101e-07,-0.000018,-0.000013


In [29]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/atac/embedding_raw.csv')

In [30]:
umap = pd.DataFrame(rna.obsm['X_umap'])
umap

,0,1
0,-6.338871,13.846613
1,-6.319490,13.824811
2,-6.571129,14.095264
3,-6.624503,14.142202
4,-6.409490,13.920228
...,...,...
2462623,12.630445,14.637726
2462624,13.079515,14.629271
2462625,4.356625,18.018282
2462626,-1.779733,11.440170


In [31]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/atac/umap_raw.csv')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_harmony

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [97]:
umap = pd.read_csv('processed/z_embedding_umap/multi_spectral/atac/umap_harmony.csv', index_col=0)
umap

,0,1
0,-9.510774,2.466275
1,-9.546992,2.511790
2,-9.370385,2.666863
3,-9.375354,2.731391
4,-9.601967,2.540223
...,...,...
2462623,14.017502,11.653357
2462624,14.071487,11.673643
2462625,-8.034444,-1.150295
2462626,6.004366,7.583872


In [98]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_raw

In [62]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [63]:
embedding = pd.DataFrame(np.load('processed/snapatac2/embedding_snap2.npy'))
embedding

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.000621,-0.000135,0.000059,-0.000080,-0.000012,0.000062,-0.000018,0.000213,0.000284,0.000157,...,0.000006,-0.000054,-0.000032,-0.000035,0.000028,-0.000057,0.000144,0.000034,-0.000022,-0.000046
1,0.000629,-0.000144,0.000068,-0.000024,0.000045,0.000070,0.000044,0.000172,0.000159,0.000168,...,-0.000026,-0.000112,-0.000028,-0.000037,0.000087,-0.000049,0.000032,0.000035,-0.000014,0.000006
2,0.000594,-0.000153,0.000101,-0.000093,-0.000017,0.000079,0.000005,0.000219,0.000246,0.000207,...,0.000039,0.000055,-0.000015,-0.000004,0.000043,-0.000020,-0.000025,0.000027,-0.000051,-0.000071
3,0.000610,-0.000147,0.000063,-0.000117,0.000036,0.000014,0.000020,0.000145,0.000256,0.000162,...,0.000019,-0.000092,-0.000025,-0.000037,0.000004,0.000015,0.000012,0.000089,0.000006,-0.000023
4,0.000609,-0.000129,0.000083,-0.000048,0.000006,0.000088,0.000022,0.000197,0.000129,0.000050,...,-0.000040,-0.000189,0.000022,-0.000025,0.000030,0.000079,-0.000007,0.000040,0.000018,-0.000036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,0.000633,-0.000017,-0.000315,0.000495,-0.000347,-0.000224,-0.000128,0.000077,0.000028,-0.000017,...,0.000003,0.000003,-0.000141,0.000235,0.000151,0.000080,0.000052,0.000045,-0.000068,0.000007
2462624,0.000814,-0.000121,-0.000165,0.000276,-0.000122,-0.000267,-0.000070,0.000051,0.000065,0.000023,...,-0.000002,-0.000025,-0.000100,0.000149,0.000049,0.000021,0.000016,0.000095,-0.000011,0.000002
2462625,0.000709,-0.000081,-0.000245,0.000362,-0.000430,-0.000246,-0.000199,-0.000050,0.000051,0.000134,...,0.000014,-0.000105,-0.000066,0.000009,0.000059,-0.000035,0.000005,-0.000026,0.000035,0.000055
2462626,0.000710,-0.000191,0.000011,-0.000070,0.000240,-0.000387,-0.000165,0.000041,-0.000043,0.000024,...,0.000006,-0.000136,-0.000067,0.000059,0.000139,-0.000075,-0.000048,0.000004,-0.000028,0.000033


In [64]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/joint/embedding_raw.csv')

In [52]:
umap = pd.DataFrame(np.load('processed/snapatac2/umap_snap2.npy'))
umap

,0,1
0,15.428403,16.526552
1,15.431728,16.507540
2,15.767015,16.914761
3,15.738565,16.888330
4,15.495610,16.592184
...,...,...
2462623,14.496370,-3.114888
2462624,14.699205,-2.976988
2462625,14.285954,-2.158833
2462626,14.399056,3.064343


In [53]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/joint/umap_raw.csv')

In [54]:
rna.obsm['X_umap'] = np.load('processed/snapatac2/umap_snap2.npy')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_multi_harmony

In [65]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [66]:
embedding = pd.DataFrame(np.load('processed/snapatac2/embedding_harmony.npy'))
embedding

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.000661,-0.000111,-0.000070,0.000028,0.000039,-0.000009,-0.000015,0.000146,6.196085e-05,-0.000019,...,0.000001,0.000023,-0.000028,-0.000006,0.000022,-0.000067,0.000144,-0.000018,-0.000016,-1.029273e-05
1,0.000669,-0.000121,-0.000057,0.000084,0.000098,-0.000002,0.000047,0.000105,-6.359823e-05,-0.000008,...,-0.000031,-0.000035,-0.000023,-0.000008,0.000081,-0.000059,0.000032,-0.000017,-0.000008,4.182524e-05
2,0.000634,-0.000129,-0.000024,0.000015,0.000036,0.000008,0.000008,0.000153,2.400466e-05,0.000031,...,0.000034,0.000132,-0.000010,0.000025,0.000037,-0.000030,-0.000025,-0.000025,-0.000045,-3.576986e-05
3,0.000650,-0.000123,-0.000064,-0.000010,0.000089,-0.000057,0.000022,0.000079,3.361513e-05,-0.000014,...,0.000014,-0.000015,-0.000020,-0.000008,-0.000002,0.000005,0.000013,0.000037,0.000012,1.259349e-05
4,0.000649,-0.000105,-0.000042,0.000060,0.000059,0.000017,0.000025,0.000131,-9.303868e-05,-0.000126,...,-0.000045,-0.000113,0.000026,0.000004,0.000024,0.000069,-0.000007,-0.000013,0.000024,-3.331905e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,0.000644,-0.000033,-0.000229,0.000387,-0.000382,-0.000167,0.000097,0.000028,-8.992545e-08,-0.000032,...,-0.000005,0.000054,-0.000063,0.000173,0.000079,0.000070,0.000023,0.000045,-0.000066,-1.565244e-06
2462624,0.000828,-0.000130,-0.000114,0.000198,-0.000260,-0.000186,0.000328,0.000005,2.460719e-05,-0.000006,...,-0.000011,0.000022,-0.000045,0.000112,-0.000014,0.000015,-0.000003,0.000086,-0.000022,-1.008416e-05
2462625,0.000720,-0.000097,-0.000160,0.000255,-0.000468,-0.000189,0.000031,-0.000099,2.307027e-05,0.000119,...,0.000006,-0.000054,0.000011,-0.000053,-0.000012,-0.000045,-0.000024,-0.000027,0.000037,4.646483e-05
2462626,0.000725,-0.000199,0.000058,-0.000145,0.000090,-0.000303,0.000254,-0.000004,-8.443770e-05,-0.000007,...,-0.000002,-0.000090,-0.000016,0.000025,0.000077,-0.000080,-0.000066,-0.000007,-0.000042,2.085259e-05


In [67]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/joint/embedding_multi_harmony.csv')

In [68]:
umap = pd.DataFrame(np.load('processed/snapatac2/umap_harmony.npy'))
umap

,0,1
0,-3.185494,-4.349708
1,-3.361973,-4.051004
2,-0.436803,-4.016902
3,-0.412247,-4.921866
4,-1.992522,-6.424223
...,...,...
2462623,-5.313932,12.010522
2462624,-3.539162,11.799864
2462625,-6.388854,13.587152
2462626,6.188811,-0.810519


In [69]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/joint/umap_multi_harmony.csv')

In [70]:
rna.obsm['X_umap'] = np.load('processed/snapatac2/umap_harmony.npy')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_harmony_multi

In [103]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue', 'level_1_colors', 'level_2_colors', 'dataset_colors', 'technology_colors'
    obsm: 'X_spectral', 'X_umap'

In [106]:
umap = pd.read_csv('processed/z_embedding_umap/multi_spectral/joint/umap_harmony_multi.csv', index_col=0)
umap

,0,1
0,15.202834,15.465776
1,15.299558,15.401444
2,16.235891,15.987560
3,16.054844,16.286633
4,15.187344,15.376781
...,...,...
2462623,15.027929,-4.980130
2462624,13.183473,-3.907983
2462625,18.695822,-1.589040
2462626,13.975087,0.290881


In [107]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_cellhint

In [77]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [78]:
embedding = pd.DataFrame(np.load('processed/snapatac2/embedding_cellhint.npy'))
embedding

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.000621,-0.000135,0.000059,-0.000080,-0.000012,0.000062,-0.000018,0.000213,0.000284,0.000157,...,0.000006,-0.000054,-0.000032,-0.000035,0.000028,-0.000057,0.000144,0.000034,-0.000022,-0.000046
1,0.000629,-0.000144,0.000068,-0.000024,0.000045,0.000070,0.000044,0.000172,0.000159,0.000168,...,-0.000026,-0.000112,-0.000028,-0.000037,0.000087,-0.000049,0.000032,0.000035,-0.000014,0.000006
2,0.000594,-0.000153,0.000101,-0.000093,-0.000017,0.000079,0.000005,0.000219,0.000246,0.000207,...,0.000039,0.000055,-0.000015,-0.000004,0.000043,-0.000020,-0.000025,0.000027,-0.000051,-0.000071
3,0.000610,-0.000147,0.000063,-0.000117,0.000036,0.000014,0.000020,0.000145,0.000256,0.000162,...,0.000019,-0.000092,-0.000025,-0.000037,0.000004,0.000015,0.000012,0.000089,0.000006,-0.000023
4,0.000609,-0.000129,0.000083,-0.000048,0.000006,0.000088,0.000022,0.000197,0.000129,0.000050,...,-0.000040,-0.000189,0.000022,-0.000025,0.000030,0.000079,-0.000007,0.000040,0.000018,-0.000036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,0.000633,-0.000017,-0.000315,0.000495,-0.000347,-0.000224,-0.000128,0.000077,0.000028,-0.000017,...,0.000003,0.000003,-0.000141,0.000235,0.000151,0.000080,0.000052,0.000045,-0.000068,0.000007
2462624,0.000814,-0.000121,-0.000165,0.000276,-0.000122,-0.000267,-0.000070,0.000051,0.000065,0.000023,...,-0.000002,-0.000025,-0.000100,0.000149,0.000049,0.000021,0.000016,0.000095,-0.000011,0.000002
2462625,0.000709,-0.000081,-0.000245,0.000362,-0.000430,-0.000246,-0.000199,-0.000050,0.000051,0.000134,...,0.000014,-0.000105,-0.000066,0.000009,0.000059,-0.000035,0.000005,-0.000026,0.000035,0.000055
2462626,0.000710,-0.000191,0.000011,-0.000070,0.000240,-0.000387,-0.000165,0.000041,-0.000043,0.000024,...,0.000006,-0.000136,-0.000067,0.000059,0.000139,-0.000075,-0.000048,0.000004,-0.000028,0.000033


In [79]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/joint/embedding_cellhint.csv')

In [80]:
umap = pd.DataFrame(np.load('processed/snapatac2/umap_cellhint.npy'))
umap

,0,1
0,9.287116,1.362543
1,10.116998,0.950386
2,8.949470,0.453466
3,8.829695,0.663664
4,10.251017,1.602941
...,...,...
2462623,4.784500,8.494054
2462624,5.139631,8.911934
2462625,5.798103,6.617845
2462626,6.780146,-2.134555


In [81]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/joint/umap_cellhint.csv')

In [82]:
rna.obsm['X_umap'] = np.load('processed/snapatac2/umap_cellhint.npy')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

## wnn

### rna_raw

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [20]:
umap = pd.read_csv('./processed/pca/rna_pca_umap.csv', index_col=0)
umap

,0,1
0,3.316055,9.871007
1,2.233050,9.771394
2,4.423573,10.241420
3,5.051735,10.431461
4,3.984083,10.966149
...,...,...
2462623,10.987378,-1.603813
2462624,11.000542,-1.309982
2462625,6.575555,-3.324849
2462626,-4.360493,6.084988


In [21]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### rna_harmony

In [26]:
umap = pd.read_csv('./processed/pca/rna_pca_harmony_umap.csv', index_col=0)
umap

,0,1
0,10.857740,7.596928
1,11.840700,6.835026
2,14.896844,6.472145
3,13.048179,9.113038
4,12.045925,4.101091
...,...,...
2462623,6.687563,12.695471
2462624,5.859000,11.486403
2462625,2.282949,14.052289
2462626,12.934424,12.771005


In [27]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_raw

In [42]:
umap = pd.read_csv('./processed/lsi/atac_lsi_umap.csv', index_col=0)
umap

,0,1
0,7.694148,-7.037178
1,8.114816,-7.562141
2,7.103957,-7.389533
3,7.126716,-8.099588
4,7.795248,-7.593944
...,...,...
2462623,2.741255,14.153102
2462624,1.740258,14.569901
2462625,10.995024,15.047639
2462626,12.854699,8.069452


In [43]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_harmony

In [49]:
umap = pd.read_csv('./processed/lsi/atac_lsi_harmony_umap.csv', index_col=0)
umap

,0,1
0,16.430414,5.813118
1,17.552052,5.128775
2,9.201924,-3.717877
3,9.268939,6.441876
4,15.780319,4.207246
...,...,...
2462623,6.781506,-3.336778
2462624,6.928165,-1.777877
2462625,5.905518,-6.319875
2462626,13.695468,13.845984


In [50]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

### atac_new

In [32]:
data = sc.read_h5ad('./processed/adata_atac_lsi_harmony.h5ad')
data

AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    uns: 'lsi', 'neighbors', 'umap'
    obsm: 'X_lsi', 'X_lsi_umap', 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'LSI'
    obsp: 'connectivities', 'distances'

In [29]:
# data.obsm['X_lsi'][:,1:])
# data.obsm['X_pca_harmony']

In [ ]:
sc.pl.umap(data, color='dataset')

In [12]:
data.obsm['X_umap'] = data.obsm['X_lsi_umap']

In [ ]:
sc.pl.umap(data, color='dataset')

### wnn_multi

In [4]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 6311
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'others', 'sample_', 'level_1', 'level_2', 'level_3'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_umap'

In [5]:
umap = pd.read_csv('./processed/wnn/umap_m.csv')
umap

,umap_1,umap_2
0,-5.078669,13.268858
1,-5.069548,12.971074
2,-5.122634,13.931956
3,-4.844479,13.719175
4,-4.782928,13.238436
...,...,...
2462623,-0.398652,-6.281715
2462624,-0.051908,-6.143399
2462625,-2.613416,-5.940579
2462626,-5.667044,-3.044367


In [6]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### wnn_harmony_multi

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [13]:
umap = pd.read_csv('./processed/wnn/umap_h_m.csv')
umap

,umap_1,umap_2
0,-6.385934,2.641150
1,-6.621700,2.297224
2,-3.940224,3.208361
3,-1.015916,3.239030
4,-5.916394,0.650503
...,...,...
2462623,7.471575,1.837388
2462624,7.477253,2.115504
2462625,6.440902,-0.446548
2462626,-0.239913,4.202368


In [14]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

# benchmarking

In [1]:
import scanpy as sc
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

2024-12-21 06:11:18.213872: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-21 06:11:18.213943: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-21 06:11:18.215508: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-21 06:11:19.147405: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


# 计算

In [2]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')

In [ ]:
rna.obsm['RNA_spectral'] = rna.obsm['X_spectral']

embedding = pd.read_csv('processed/multi_spectral/rna/embedding_harmony.csv', index_col=0).to_numpy()
rna.obsm['RNA_spectral_harmony'] = embedding

embedding = pd.read_csv('processed/multi_spectral/atac/embedding_raw.csv', index_col=0).to_numpy()
rna.obsm['ATAC_spectral'] = embedding

embedding = pd.read_csv('processed/multi_spectral/atac/embedding_harmony.csv', index_col=0).to_numpy()
rna.obsm['ATAC_spectral_harmony'] = embedding

embedding = pd.read_csv('processed/multi_spectral/joint/embedding_harmony_multi.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral_harmony_multi'] = embedding

In [ ]:
embedding = pd.read_csv('processed/multi_spectral/joint/embedding_raw.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral'] = embedding

embedding = pd.read_csv('processed/multi_spectral/joint/embedding_multi_harmony.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral_multi_harmony'] = embedding'

embedding = pd.read_csv('processed/multi_spectral/joint/embedding_cellhint.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral_multi_cellhint'] = embedding

embedding = pd.read_csv('../7_horizontal_RNA_ATAC/results/blood/uinmf_latent_k30.csv', header=0, index_col=0).to_numpy()
rna.obsm['joint_uinmf'] = embedding

embedding = pd.read_csv('../7_horizontal_RNA_ATAC/results/blood/latent_mtv.csv', header=None).to_numpy()
rna.obsm['joint_multivi'] = embedding

In [3]:
embedding = pd.read_csv('processed/pca/rna_pca.csv', index_col=0).to_numpy()
rna.obsm['RNA_pca'] = embedding

embedding = pd.read_csv('processed/pca/rna_pca_harmony.csv', index_col=0).to_numpy()
rna.obsm['RNA_pca_harmony'] = embedding

embedding = pd.read_csv('processed/lsi/atac_lsi.csv', index_col=0).to_numpy()
rna.obsm['ATAC_lsi'] = embedding

embedding = pd.read_csv('processed/lsi/atac_lsi_harmony.csv', index_col=0).to_numpy()
rna.obsm['ATAC_lsi_harmony'] = embedding

# umap = pd.read_csv('./processed/wnn/umap_m.csv').to_numpy()
# rna.obsm['joint_wnn'] = umap

# umap = pd.read_csv('./processed/wnn/umap_h_m.csv').to_numpy()
# rna.obsm['joint_harmony_wnn'] = umap

In [ ]:
bm = Benchmarker(
    rna,
    batch_key="sample",
    label_key="level_3",
    embedding_obsm_keys=['RNA_spectral','RNA_spectral_harmony'],
    n_jobs=-1,
)
bm.benchmark()

/home/qiaolibo/.local/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:377: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)
Computing neighbors:  50%|█████████████████████████████████▌                                 | 1/2 [37:11<37:11, 2231.53s/it]

In [ ]:
bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_results_table.pdf", dpi=300)

In [ ]:
df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_results.csv", index=True)

In [ ]:
# 保存 Benchmarker 对象
with open("benchmarker.pkl", "wb") as f:
    pickle.dump(bm, f)

# 加载 Benchmarker 对象
with open("benchmarker.pkl", "rb") as f:
    bm_loaded = pickle.load(f)